# Sprint 4: RL Training Dynamics

This notebook has two modes:
- **Pre-run (now):** Simulates expected training curves using synthetic data to validate
  monitoring logic and set convergence expectations before committing GPU time.
- **Post-run:** Replace `SYNTHETIC = True` with `SYNTHETIC = False` and point
  `TRAJECTORY_DIR` at real checkpoint trajectories to plot actual results.

Metrics tracked per iteration:
1. Solve rate (binary — did expression evaluate to 24?)
2. Average shaped reward (format + numbers + solve components)
3. Partial-credit rate (trajectories with 0 < reward < 1)
4. Reward component breakdown (what the model is learning)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

from src.rl.rewards import compute_reward, FORMAT_REWARD, NUMBERS_REWARD, SOLVE_REWARD
from src.rl.trajectory import TrajectoryBuffer

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# ── Configuration ─────────────────────────────────────────────────────────────
SYNTHETIC = True          # set False after a real GPU run
TRAJECTORY_DIR = Path('../checkpoints/grpo')  # real trajectory path
N_ITERATIONS = 5
ROLLOUTS_PER_ITER = 50
RANDOM_SEED = 42
# ──────────────────────────────────────────────────────────────────────────────

## 1. Load or Simulate Trajectory Data

In [ ]:
def simulate_training_curve(n_iter, rollouts, seed):
    """Synthesise plausible training dynamics for planning purposes.
    
    Models a realistic GRPO learning curve:
    - Solve rate starts at ~30% (LLM zero-shot CoT baseline)
    - Shaped reward rises faster than solve rate (format/numbers components)
    - Convergence slows after iteration 3 (easy puzzles saturate)
    """
    rng = random.Random(seed)
    
    # Expected solve rate trajectory (S-curve)
    base_rates = [0.30, 0.42, 0.54, 0.61, 0.65]
    
    iterations = []
    for i in range(n_iter):
        solve_rate = base_rates[i] + rng.gauss(0, 0.02)
        solve_rate = max(0.0, min(1.0, solve_rate))
        
        # Partial credit appears before solve rate climbs
        partial_rate = max(0.0, 0.25 - i * 0.03 + rng.gauss(0, 0.015))
        
        n_solved   = int(solve_rate * rollouts)
        n_partial  = int(partial_rate * rollouts)
        n_zero     = rollouts - n_solved - n_partial
        
        rewards = (
            [SOLVE_REWARD] * n_solved +
            [FORMAT_REWARD + NUMBERS_REWARD] * n_partial +
            [0.0] * max(0, n_zero)
        )
        
        iterations.append({
            'iteration': i + 1,
            'solve_rate': n_solved / rollouts,
            'avg_reward': sum(rewards) / len(rewards),
            'partial_rate': n_partial / rollouts,
            'zero_rate': max(0, n_zero) / rollouts,
            'n_solved': n_solved,
            'n_partial': n_partial,
            'n_zero': max(0, n_zero),
        })
    return iterations


def load_real_trajectories(trajectory_dir, n_iter):
    """Load actual trajectory JSONL files from a completed training run."""
    iterations = []
    for i in range(n_iter):
        path = trajectory_dir / f'trajectories_iter{i:03d}.jsonl'
        if not path.exists():
            print(f'Missing: {path}')
            continue
        buf = TrajectoryBuffer.load(path)
        all_t = buf.all()
        rewards = [t.reward for t in all_t]
        iterations.append({
            'iteration': i + 1,
            'solve_rate': sum(t.solved for t in all_t) / len(all_t),
            'avg_reward': sum(rewards) / len(rewards),
            'partial_rate': sum(0 < r < 1 for r in rewards) / len(rewards),
            'zero_rate': sum(r == 0 for r in rewards) / len(rewards),
            'n_solved': sum(t.solved for t in all_t),
            'n_partial': sum(0 < t.reward < 1 for t in all_t),
            'n_zero': sum(t.reward == 0 for t in all_t),
        })
    return iterations


if SYNTHETIC:
    print('Mode: SYNTHETIC (simulated training curve)')
    data = simulate_training_curve(N_ITERATIONS, ROLLOUTS_PER_ITER, RANDOM_SEED)
else:
    print(f'Mode: REAL (loading from {TRAJECTORY_DIR})')
    data = load_real_trajectories(TRAJECTORY_DIR, N_ITERATIONS)

for d in data:
    print(f"Iter {d['iteration']}: solve={d['solve_rate']:.1%}  "
          f"avg_reward={d['avg_reward']:.3f}  "
          f"partial={d['partial_rate']:.1%}  zero={d['zero_rate']:.1%}")

## 2. Solve Rate & Shaped Reward per Iteration

In [ ]:
iters       = [d['iteration']    for d in data]
solve_rates = [d['solve_rate']   for d in data]
avg_rewards = [d['avg_reward']   for d in data]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Solve rate
ax1.plot(iters, solve_rates, 'o-', color='#4e79a7', linewidth=2, markersize=7)
ax1.axhline(0.77, color='#e15759', linestyle='--', linewidth=1.2, label='Brute-force ceiling (77%)')
ax1.axhline(0.58, color='#f28e2b', linestyle='--', linewidth=1.2, label='MCTS random baseline (58%)')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax1.set_xlabel('Training Iteration')
ax1.set_ylabel('Solve Rate')
ax1.set_title('Solve Rate per Iteration' + (' (synthetic)' if SYNTHETIC else ''))
ax1.legend(fontsize=9)
ax1.set_ylim(0, 0.9)
ax1.set_xticks(iters)
ax1.grid(axis='y', alpha=0.3)

# Avg shaped reward
ax2.plot(iters, avg_rewards, 's-', color='#59a14f', linewidth=2, markersize=7)
ax2.axhline(1.0, color='#e15759', linestyle='--', linewidth=1.2, label='Max reward (1.0)')
ax2.set_xlabel('Training Iteration')
ax2.set_ylabel('Average Shaped Reward')
ax2.set_title('Avg Shaped Reward per Iteration' + (' (synthetic)' if SYNTHETIC else ''))
ax2.set_ylim(0, 1.05)
ax2.set_xticks(iters)
ax2.grid(axis='y', alpha=0.3)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../docs/api/fig_training_curves.png', bbox_inches='tight')
plt.show()

print(f'\nFinal iteration: solve rate = {solve_rates[-1]:.1%}, avg reward = {avg_rewards[-1]:.3f}')

## 3. Reward Component Breakdown — Where Is the Model Learning?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

n_solved  = [d['n_solved']  for d in data]
n_partial = [d['n_partial'] for d in data]
n_zero    = [d['n_zero']    for d in data]
total     = ROLLOUTS_PER_ITER

bars_zero    = ax.bar(iters, n_zero,    color='#e15759', label='Zero reward (no tags / wrong numbers)')
bars_partial = ax.bar(iters, n_partial, bottom=n_zero,   color='#f28e2b', label='Partial credit (format + numbers)')
bars_solved  = ax.bar(iters, n_solved,  bottom=[z+p for z,p in zip(n_zero,n_partial)],
                      color='#4e79a7', label='Solved (reward = 1.0)')

ax.set_xlabel('Training Iteration')
ax.set_ylabel('Trajectories per Iteration')
ax.set_title('Trajectory Composition per Iteration' + (' (synthetic)' if SYNTHETIC else ''))
ax.set_xticks(iters)
ax.legend(loc='upper left', fontsize=9)
ax.axhline(total, color='black', linewidth=0.8, linestyle=':')
ax.set_ylim(0, total * 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/api/fig_reward_composition.png', bbox_inches='tight')
plt.show()

## 4. GRPO Gradient Flow Health Check

GRPO needs within-group reward variance to produce non-zero gradients.
Groups where all rewards are identical (all-0 or all-1) contribute zero gradient.

A healthy training run should show group variance > 0 at every iteration.

In [ ]:
GROUP_SIZE = 8

def estimate_group_variance(solve_rate, partial_rate, group_size):
    """Monte Carlo estimate of avg within-group reward variance."""
    rng = random.Random(42)
    zero_rate = max(0.0, 1.0 - solve_rate - partial_rate)
    
    reward_dist = (
        [1.0]   * int(solve_rate   * 1000) +
        [FORMAT_REWARD + NUMBERS_REWARD] * int(partial_rate * 1000) +
        [0.0]   * int(zero_rate    * 1000)
    )
    if not reward_dist:
        return 0.0
    
    variances = []
    for _ in range(500):
        group = rng.choices(reward_dist, k=group_size)
        mean = sum(group) / group_size
        var = sum((r - mean)**2 for r in group) / group_size
        variances.append(var)
    return sum(variances) / len(variances)

group_vars = [
    estimate_group_variance(d['solve_rate'], d['partial_rate'], GROUP_SIZE)
    for d in data
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(iters, group_vars, color='#76b7b2', width=0.5)
ax.axhline(0.0, color='red', linewidth=1, linestyle='--', label='Zero variance = no gradient')
ax.set_xlabel('Training Iteration')
ax.set_ylabel('Avg Within-Group Reward Variance')
ax.set_title(f'GRPO Gradient Flow Health (group_size={GROUP_SIZE})' + (' — synthetic' if SYNTHETIC else ''))
ax.set_xticks(iters)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/api/fig_group_variance.png', bbox_inches='tight')
plt.show()

for i, (d, v) in enumerate(zip(data, group_vars)):
    health = 'GOOD' if v > 0.05 else 'LOW — check reward distribution'
    print(f"Iter {d['iteration']}: group variance = {v:.4f}  [{health}]")

## 5. Convergence Summary

In [ ]:
MCTS_BASELINE = 0.58
BRUTE_FORCE_CEILING = 0.77

final = data[-1]
gap_closed = (final['solve_rate'] - MCTS_BASELINE) / (BRUTE_FORCE_CEILING - MCTS_BASELINE)

print('=' * 50)
print('Sprint 4 Convergence Summary')
print('=' * 50)
print(f"  MCTS random rollout baseline : {MCTS_BASELINE:.1%}")
print(f"  Brute-force ceiling          : {BRUTE_FORCE_CEILING:.1%}")
print(f"  Final solve rate ({N_ITERATIONS} iters)   : {final['solve_rate']:.1%}")
print(f"  Gap closed vs. MCTS baseline : {gap_closed:.1%}")
print(f"  Final avg shaped reward      : {final['avg_reward']:.3f}")
print()
print('Status:', 'synthetic projection' if SYNTHETIC else 'real GPU run')